In [1]:
!pip install librosa numpy pandas scikit-learn tensorflow tqdm

In [2]:
import os, json
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras import layers, models


In [3]:
import kagglehub
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")

Using Colab cache for faster access to the 'asvpoof-2019-dataset' dataset.


In [4]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=16000, duration=2)
        spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
        log_spec = librosa.power_to_db(spec)
        return log_spec
    except:
        return None


In [11]:
# ASVspoof loading
import os

# Dynamically search for the required directory and file within the downloaded KaggleHub path
asv_path = None
label_file = None

for root, dirs, files in os.walk(path):
    if 'ASVspoof2019_LA_train' in root and 'flac' in dirs:
        asv_path = os.path.join(root, 'flac')
    for f in files:
        if f == 'ASVspoof2019.LA.cm.train.trn.txt':
            label_file = os.path.join(root, f)

if asv_path is None or label_file is None:
    raise FileNotFoundError(
        f"Could not find required dataset directories under path: {path}. "
        f"Detected paths: asv_path={asv_path}, label_file={label_file}"
    )

print(f"Found audio path at: {asv_path}")
print(f"Found label file at: {label_file}")

cols = ['speaker','file','env','attack','label']
df = pd.read_csv(label_file, sep=' ', names=cols)

X_asv, y_asv = [], []
# Extract features for a subset to save time
for _, row in tqdm(df.iterrows(), total=200):
    file_path = os.path.join(asv_path, row['file'] + '.flac')
    if os.path.exists(file_path):
        feat = extract_features(file_path)
        if feat is not None:
            X_asv.append(feat)
            y_asv.append(1 if row['label']=='spoof' else 0)

print(f"Loaded {len(X_asv)} ASVspoof samples.")

Found audio path at: /kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac
Found label file at: /kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt


25380it [07:46, 54.35it/s]

Loaded 25380 ASVspoof samples.


In [12]:
# FakeAVCeleb loading
import os
import json
from tqdm import tqdm

fakeav_path = '/content/FakeAVCeleb/videos/'
meta_path = '/content/FakeAVCeleb/metadata.json'

X_fake, y_fake = [], []

if os.path.exists(meta_path):
    with open(meta_path) as f:
        metadata = json.load(f)

    for item in tqdm(metadata[:2000]):
        file_path = os.path.join(fakeav_path, item['file'])
        if os.path.exists(file_path):
            feat = extract_features(file_path)
            if feat is not None:
                X_fake.append(feat)
                y_fake.append(1 if item['audio_label']=='fake' else 0)
else:
    print(f"Warning: {meta_path} not found. Skipping FakeAVCeleb dataset and training with ASVspoof data only.")

In [14]:
# Fix inhomogeneous shapes by padding/truncating all features to a target length of 63 along the time axis
target_time_steps = 63
processed_features = []

for feat in (X_asv + X_fake):
    if feat.shape[1] < target_time_steps:
        # Pad with minimum value (silence in dB)
        pad_width = target_time_steps - feat.shape[1]
        feat = np.pad(feat, ((0, 0), (0, pad_width)), mode='minimum')
    elif feat.shape[1] > target_time_steps:
        # Truncate to target length
        feat = feat[:, :target_time_steps]
    processed_features.append(feat)

X = np.array(processed_features)
y = np.array(y_asv + y_fake)
X = X[..., np.newaxis]
X, y = shuffle(X, y, random_state=42)
print('Total samples:', len(X))
print('Shape of X:', X.shape)

Total samples: 25380
Shape of X: (25380, 128, 63, 1)


In [15]:
model = models.Sequential([
    layers.Conv2D(32,(3,3),activation='relu',input_shape=X.shape[1:]),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128,(3,3),activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1,activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 61, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 61, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 6, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,209 (430.50 KB)

 Trainable params: 109,761 (428.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))


Epoch 1/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 37s 35ms/step - accuracy: 0.9189 - loss: 0.2048 - val_accuracy: 0.9330 - val_loss: 0.1706
Epoch 2/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9628 - loss: 0.0988 - val_accuracy: 0.9301 - val_loss: 0.1518
Epoch 3/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9769 - loss: 0.0630 - val_accuracy: 0.9086 - val_loss: 0.6093
Epoch 4/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9845 - loss: 0.0433 - val_accuracy: 0.9791 - val_loss: 0.0517
Epoch 5/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9882 - loss: 0.0350 - val_accuracy: 0.9521 - val_loss: 0.1423
Epoch 6/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9898 - loss: 0.0292 - val_accuracy: 0.9525 - val_loss: 0.1300
Epoch 7/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9926 - loss: 0.0215 - val_accuracy: 0.9815 - val_loss: 0.0441
Epoch 8/10
635/635 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9934 - loss: 0.0194 - val_acc

In [17]:
model.save_weights('audio.weights.h5')
